> **Disclaimer:** This workshop is provided for educational and informational purposes only. The sample code and configurations are not intended for production use. Please review and adapt them according to your organization's security and compliance requirements. AWS service charges may apply for resources created during this workshop. Video content used in this workshop is sourced from publicly available materials and is attributed where applicable.

# Approach 2: Multi-Vector Retrieval with Fixed Weights

This notebook keeps visual, audio, and transcription embeddings **separate** and combines similarity scores at query time.

Two ranking strategies:
1. **Score-based fusion:** `score(s) = Σ w_m * sim(Q, E_m(s))`
2. **Reciprocal Rank Fusion (RRF):** `score_RRF(s) = Σ 1/(k + rank_m(s))`

In [ ]:
import json
import numpy as np
import boto3
from botocore.config import Config
from collections import defaultdict

# Auto-load configuration from 01_setup_and_embedding.ipynb
with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
MODEL_ID_SYNC = config["MODEL_ID_SYNC"]
VECTOR_BUCKET = config["VECTOR_BUCKET"]
DATA_S3_PREFIX = config["DATA_S3_PREFIX"]
RESULTS_S3_PREFIX = config["RESULTS_S3_PREFIX"]

print(f"✅ Configuration loaded from config.json")
print(f"   S3 Bucket: {S3_BUCKET}")
print(f"   Vector Bucket: {VECTOR_BUCKET}")

bedrock = boto3.client("bedrock-runtime", region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client("s3", region_name=REGION)
s3v = boto3.client("s3vectors", region_name=REGION)

def fmt_time(sec):
    """Convert seconds to mm:ss format"""
    m, s = divmod(sec, 60)
    return f"{int(m):02d}:{s:04.1f}"

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed_text(query):
    response = bedrock.invoke_model(
        modelId=MODEL_ID_SYNC,
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    return json.loads(response["body"].read())["data"][0]["embedding"]

print("Setup complete")

## 1. Load Separate Modality Indices

In [ ]:
# Load embeddings from S3
indices = {}
for modality in ["visual", "audio", "transcription"]:
    s3_key = f"{DATA_S3_PREFIX}embeddings_{modality}.json"
    obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    indices[modality] = json.loads(obj["Body"].read().decode("utf-8"))
    print(f"{modality}: {len(indices[modality])} segments")

## 2. Score-based Multi-Vector Search

Search each index separately, then combine scores with modality weights.

In [ ]:
W_V, W_A, W_T = 0.8, 0.1, 0.05
WEIGHTS = {"visual": W_V, "audio": W_A, "transcription": W_T}

def query_modality_s3v(query_embedding, modality, top_k=20):
    """Search a single modality index in S3 Vectors"""
    response = s3v.query_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=modality,
        topK=top_k,
        queryVector={"float32": [float(x) for x in query_embedding]},
        returnMetadata=True,
        returnDistance=True
    )
    results = []
    for v in response["vectors"]:
        results.append({
            "startSec": v["metadata"]["startSec"],
            "endSec": v["metadata"]["endSec"],
            "similarity": v["distance"]
        })
    return results

def score_based_fusion(query_embedding, weights, top_k=5):
    """Multi-vector search with score-based fusion using S3 Vectors"""
    modality_results = {}
    for modality in weights:
        modality_results[modality] = query_modality_s3v(query_embedding, modality, top_k=50)
    
    combined = defaultdict(lambda: {"scores": {}, "startSec": 0, "endSec": 0})
    for modality, results in modality_results.items():
        for r in results:
            key = (round(r["startSec"], 1), round(r["endSec"], 1))
            combined[key]["scores"][modality] = r["similarity"]
            combined[key]["startSec"] = r["startSec"]
            combined[key]["endSec"] = r["endSec"]
    
    final_results = []
    for key, data in combined.items():
        weighted_score = sum(
            weights.get(m, 0) * data["scores"].get(m, 0)
            for m in weights
        )
        final_results.append({
            "startSec": data["startSec"],
            "endSec": data["endSec"],
            "combined_score": weighted_score,
            "visual_score": data["scores"].get("visual", 0),
            "audio_score": data["scores"].get("audio", 0),
            "transcription_score": data["scores"].get("transcription", 0)
        })
    
    final_results.sort(key=lambda x: x["combined_score"], reverse=True)
    return final_results[:top_k]

print("Score-based fusion with S3 Vectors ready")

## 3. Reciprocal Rank Fusion (RRF)

`score_RRF(s) = Σ 1/(k + rank_m(s))` where k=60

In [ ]:
def rrf_fusion(query_embedding, k=60, top_k=5):
    """Multi-vector search with Reciprocal Rank Fusion using S3 Vectors"""
    modality_rankings = {}
    for modality in WEIGHTS:
        results = query_modality_s3v(query_embedding, modality, top_k=50)
        rankings = {}
        for rank, r in enumerate(results, 1):
            key = (round(r["startSec"], 1), round(r["endSec"], 1))
            rankings[key] = rank
        modality_rankings[modality] = rankings
    
    all_segments = set()
    for rankings in modality_rankings.values():
        all_segments.update(rankings.keys())
    
    rrf_results = []
    for seg_key in all_segments:
        rrf_score = 0
        ranks = {}
        for modality, rankings in modality_rankings.items():
            rank = rankings.get(seg_key, 1000)
            rrf_score += 1.0 / (k + rank)
            ranks[modality] = rank
        
        rrf_results.append({
            "startSec": seg_key[0],
            "endSec": seg_key[1],
            "rrf_score": rrf_score,
            "visual_rank": ranks.get("visual", 999),
            "audio_rank": ranks.get("audio", 999),
            "transcription_rank": ranks.get("transcription", 999)
        })
    
    rrf_results.sort(key=lambda x: x["rrf_score"], reverse=True)
    return rrf_results[:top_k]

print("RRF fusion with S3 Vectors ready")

## 4. Test with Multiple Queries

In [ ]:
TEST_QUERIES = [
    {"query": "a player kicking the ball towards the goal", "intent": "visual"},
    {"query": "crowd cheering and celebrating loudly", "intent": "audio"},
    {"query": "the commentator describes the player's performance", "intent": "transcription"},
    {"query": "a goal being scored with the stadium erupting", "intent": "visual+audio"},
    {"query": "the announcer explaining a foul while the crowd boos", "intent": "mixed"},
    {"query": "players running on the soccer field", "intent": "visual"},
    {"query": "referee whistle blowing during the match", "intent": "audio"},
    {"query": "the commentator discusses the team's strategy", "intent": "transcription"},
]

score_results_all = {}
rrf_results_all = {}

for item in TEST_QUERIES:
    query = item["query"]
    intent = item["intent"]
    
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\" (intent: {intent})")
    print(f"{'='*70}")
    
    # Embed query ONCE and reuse for both methods
    q_emb = embed_text(query)
    
    # Score-based fusion via S3 Vectors
    score_res = score_based_fusion(q_emb, WEIGHTS)
    score_results_all[query] = score_res
    
    print("\n  [Score-based Fusion]")
    for i, r in enumerate(score_res[:5], 1):
        print(f"  {i}. [{fmt_time(r['startSec'])}-{fmt_time(r['endSec'])}] "
              f"combined={r['combined_score']:.4f} "
              f"(V={r['visual_score']:.3f} A={r['audio_score']:.3f} T={r['transcription_score']:.3f})")
    
    # RRF via S3 Vectors (reuse same embedding)
    rrf_res = rrf_fusion(q_emb)
    rrf_results_all[query] = rrf_res
    
    print("\n  [RRF Fusion (k=60)]")
    for i, r in enumerate(rrf_res[:5], 1):
        print(f"  {i}. [{fmt_time(r['startSec'])}-{fmt_time(r['endSec'])}] "
              f"rrf={r['rrf_score']:.6f} "
              f"(V_rank={r['visual_rank']} A_rank={r['audio_rank']} T_rank={r['transcription_rank']})")

## 5. Debuggability Advantage

Unlike fused embeddings, we can now see **exactly which modality contributed** to each result.

In [ ]:
print("Per-Modality Score Breakdown (Score-based Fusion)")
print("=" * 80)

for item in TEST_QUERIES:
    query = item["query"]
    intent = item["intent"]
    results = score_results_all[query]
    
    if not results:
        continue
    
    print(f"\nQuery: \"{query}\" (intent: {intent})")
    print(f"  Top result: [{fmt_time(results[0]['startSec'])}-{fmt_time(results[0]['endSec'])}]")
    
    top = results[0]
    total = top["combined_score"]
    if total > 0:
        v_pct = (WEIGHTS["visual"] * top["visual_score"] / total) * 100
        a_pct = (WEIGHTS["audio"] * top["audio_score"] / total) * 100
        t_pct = (WEIGHTS["transcription"] * top["transcription_score"] / total) * 100
        print(f"  Visual contribution:        {v_pct:5.1f}%")
        print(f"  Audio contribution:         {a_pct:5.1f}%")
        print(f"  Transcription contribution: {t_pct:5.1f}%")
        
        dominant = max([("visual", v_pct), ("audio", a_pct), ("transcription", t_pct)], key=lambda x: x[1])
        print(f"  >> Dominant modality: {dominant[0]} ({dominant[1]:.1f}%)")

print("\n\nRRF vs Score-based: Top-1 Agreement")
print("=" * 80)
agree = 0
for item in TEST_QUERIES:
    query = item["query"]
    score_top = score_results_all[query][0] if score_results_all[query] else None
    rrf_top = rrf_results_all[query][0] if rrf_results_all[query] else None
    
    if score_top and rrf_top:
        s_key = (round(score_top["startSec"], 1), round(score_top["endSec"], 1))
        r_key = (round(rrf_top["startSec"], 1), round(rrf_top["endSec"], 1))
        match = s_key == r_key
        if match:
            agree += 1
        status = "AGREE" if match else "DIFFER"
        print(f"  [{status}] \"{query[:50]}\"")
        if not match:
            print(f"           Score: [{fmt_time(score_top['startSec'])}-{fmt_time(score_top['endSec'])}], "
                  f"RRF: [{fmt_time(rrf_top['startSec'])}-{fmt_time(rrf_top['endSec'])}]")

print(f"\nAgreement: {agree}/{len(TEST_QUERIES)}")

In [ ]:
# Save results to S3
results_data = {
    "approach": "multivector_fixed_weights",
    "weights": WEIGHTS,
    "score_results": {q: r for q, r in score_results_all.items()},
    "rrf_results": {q: [dict(r) for r in rs] for q, rs in rrf_results_all.items()}
}

s3_key = f"{RESULTS_S3_PREFIX}multivector_fixed_results.json"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=s3_key,
    Body=json.dumps(results_data, indent=2, default=float),
    ContentType="application/json"
)

print(f"Results saved to s3://{S3_BUCKET}/{s3_key}")

---

## Summary: Multi-Vector Fixed Weights Approach

### Architecture
```
[Stored in Notebook 01]
  S3 Vectors: visual index (161 vectors, 512-dim)
  S3 Vectors: audio index (161 vectors, 512-dim)
  S3 Vectors: transcription index (161 vectors, 512-dim)

[Search query] -> embed_text() -> 1 query vector
  |-- visual index search       -> Top-50 results + similarity scores
  |-- audio index search        -> Top-50 results + similarity scores
  |-- transcription index search -> Top-50 results + similarity scores
       |
       |-- 2 combination methods:
           |-- Score-based: 0.8 x V_score + 0.1 x A_score + 0.05 x T_score
           |-- RRF: 1/(60+V_rank) + 1/(60+A_rank) + 1/(60+T_rank)
```

### Key Difference from Notebook 02 (Fused)
```
Notebook 02: Merged at storage time -> too late at query time (irreversible)
Notebook 03: Stored separately      -> merged at query time (reversible, weights can change freely)
```

### Score-based Fusion
- Formula: `score = w_v x sim(Q, V) + w_a x sim(Q, A) + w_t x sim(Q, T)`
- Weights: visual=0.8, audio=0.1, transcription=0.05 (fixed)
- Pros: Can prioritize specific modalities
- Cons: Score scales may differ across modalities (unfair), weights are fixed

### RRF (Reciprocal Rank Fusion)
- Formula: `score = 1/(k + rank_visual) + 1/(k + rank_audio) + 1/(k + rank_transcription)`
- k=60 (constant that smooths rank differences)
- Pros: No weight tuning needed, scale-agnostic, treats all 3 modalities equally
- Cons: Cannot prioritize a specific modality

### What This Notebook Verifies

**1. Debuggability (cell-11)**
```
Query: "crowd cheering and celebrating loudly" (intent: audio)
  Visual contribution:   72.3%   <- Still visual-dominant due to 0.8 weight
  Audio contribution:    18.5%
  Transcription:          9.2%
```
-> The problem of "audio query but visual dominates" is now **visible in numbers** (impossible in Notebook 02)

**2. Score-based vs RRF Result Comparison**
- Check whether Top-1 results agree or differ for the same query
- Visual intent queries: Score-based has advantage (80% visual weight)
- Audio/transcription intent queries: RRF may be more fair

### Remaining Limitations
- Score-based: Weights are still **fixed** -> visual 80% even for audio queries
- RRF: Has **no weights** -> cannot specify modality priority
- Neither **adapts to query intent**

### Next Step
-> Notebook 04 solves this with Intent-based Dynamic Routing that automatically computes per-query weights